# 2 — Score the samples

Every metric reported in the paper, computed on the arrays notebook 1 wrote,
followed by four views that compare the generated distribution to the real one.

Nothing here retrains the generator. It reads
`results/run/<dataset>_sfmtime/<dataset>_sfmtime_data.npz`, so a competing
method that writes the same file with the same keys is scored by exactly this
code.

In [1]:
import os
# torch and scikit-learn each bring their own OpenMP runtime; inside a Jupyter
# kernel the two can collide and take the kernel down when t-SNE runs after a
# torch import. Capping the thread pools before either is imported avoids it.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import sys, json
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import torch
import matplotlib.pyplot as plt

from sfmtime.metrics import (discriminative_score, predictive_score, correlation_score,
                             fit_context_encoder, context_fid, delta_distance,
                             spectral_distance, acf_distance)

plt.rcParams.update({"figure.dpi": 120, "font.size": 8,
                     "axes.spines.top": False, "axes.spines.right": False})
REAL_C, GEN_C = "#A82424", "#17548C"

In [2]:
DATASET = "sines"      # must match notebook 1
REPEATS = 10           # evaluator seeds for the learned metrics
CFID_REPEATS = 3       # encoder fits behind Context-FID; each one refits TS2Vec

RUN = os.path.join("results", "run", f"{DATASET}_sfmtime")
z = np.load(os.path.join(RUN, f"{DATASET}_sfmtime_data.npz"))
real_train, real_test, gen = z["real_train"], z["real_test"], z["generated"]
run = json.load(open(os.path.join(RUN, "run.json")))
print(f"{DATASET}: real_test {real_test.shape}  generated {gen.shape}")
print(f"NFE {run['nfe']}  |  {run['train_minutes']} min to train  "
      f"|  {run['sample_ms_per_sequence']} ms per sequence")

sines: real_test (1000, 64, 4)  generated (3000, 64, 4)
NFE 80  |  18.58 min to train  |  47.744 ms per sequence


## Why the learned metrics are repeated

The discriminative score trains a fresh GRU to separate real from generated, so
it carries real estimator variance: on *fixed* data it moves by enough across
evaluator seeds that a single-seed number can be read as a large difference when
nothing has changed. Any ranking taken from one seed is mostly reading noise, so
these are reported as mean ± standard deviation over `REPEATS` seeds.

In [3]:
disc = [discriminative_score(real_test, gen, seed=s) for s in range(REPEATS)]
pred = [predictive_score(real_test, gen, seed=s) for s in range(REPEATS)]
print(f"discriminative  {np.mean(disc):.4f} +- {np.std(disc):.4f}   "
      f"(range {min(disc):.4f}-{max(disc):.4f})")
print(f"predictive      {np.mean(pred):.4f} +- {np.std(pred):.4f}")

discriminative  0.0232 +- 0.0175   (range 0.0050-0.0575)
predictive      0.0102 +- 0.0002


Context-FID is a Fréchet distance in the representation space of a TS2Vec
encoder fitted on the real sequences. Each repeat refits the encoder, so this is
the expensive dial.

In [ ]:
cf = []
for s in range(CFID_REPEATS):
    enc = fit_context_encoder(torch.as_tensor(real_test).float(), device="cpu", seed=s)
    cf.append(float(context_fid(torch.as_tensor(real_test).float(),
                                torch.as_tensor(gen).float(), enc, device="cpu")))
print(f"context-FID     {np.mean(cf):.4f} +- {np.std(cf):.4f}")

The remaining four are deterministic functions of the two sets, so they carry no
evaluator seed: correlational (cross-channel structure), increment (step-to-step
differences), autocorrelation and spectral distance.

In [ ]:
det = {
    "correlational":   float(correlation_score(real_test, gen)),
    "increment":       float(delta_distance(real_test, gen)),
    "autocorrelation": float(acf_distance(real_test, gen)),
    "spectral":        float(spectral_distance(real_test, gen)),
}
for k, v in det.items():
    print(f"{k:16s}{v:.4f}")

scores = {"discriminative": {"mean": float(np.mean(disc)), "std": float(np.std(disc))},
          "predictive":     {"mean": float(np.mean(pred)), "std": float(np.std(pred))},
          "context_fid":    {"mean": float(np.mean(cf)),   "std": float(np.std(cf))},
          **{k: {"mean": v, "std": 0.0} for k, v in det.items()}}
json.dump(scores, open(os.path.join(RUN, "metrics.json"), "w"), indent=2)

## Summary

Lower is better on all seven. The discriminative score is $|0.5 - \mathrm{acc}|$
of a classifier trained to tell the two sets apart, so $0$ means it cannot.

In [ ]:
print(f"{'metric':18s}{'value':>10s}{'std':>9s}")
print("-" * 37)
for k, v in scores.items():
    sd = f"{v['std']:.4f}" if v["std"] else "--"
    print(f"{k:18s}{v['mean']:>10.4f}{sd:>9s}")

## Where the two distributions agree and differ

Four views, each answering a different question. PCA and t-SNE ask whether the
generated sequences occupy the same region of sequence space; the value density
asks whether they have the same marginal, without reference to temporal order;
the power spectrum asks whether they have the same temporal structure.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

k = min(600, len(real_test), len(gen))
rng = np.random.default_rng(0)
r = real_test[rng.choice(len(real_test), k, replace=False)].reshape(k, -1)
g = gen[rng.choice(len(gen), k, replace=False)].reshape(k, -1)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.6))

pc = PCA(n_components=2, random_state=0).fit(r)
for arr, c, lab in ((pc.transform(r), REAL_C, "real"), (pc.transform(g), GEN_C, "generated")):
    axes[0].scatter(arr[:, 0], arr[:, 1], s=4, c=c, alpha=.4, lw=0, label=lab)
axes[0].set_title("PCA"); axes[0].legend(frameon=False, fontsize=7)

# t-SNE has no out-of-sample transform, so both sets are embedded together and
# the panel draws the two halves of one embedding.
Z = PCA(n_components=min(50, r.shape[1], 2 * k - 1), random_state=0).fit_transform(
    np.concatenate([r, g]))
emb = TSNE(n_components=2, perplexity=30, init="pca",
           learning_rate="auto", random_state=0).fit_transform(Z)
axes[1].scatter(emb[:k, 0], emb[:k, 1], s=4, c=REAL_C, alpha=.4, lw=0)
axes[1].scatter(emb[k:, 0], emb[k:, 1], s=4, c=GEN_C, alpha=.4, lw=0)
axes[1].set_title("t-SNE")

from scipy.stats import gaussian_kde
lo = min(np.percentile(real_test, 0.05), np.percentile(gen, 0.5))
hi = max(np.percentile(real_test, 99.95), np.percentile(gen, 99.5))
grid = np.linspace(lo, hi, 300)
for arr, c, ls in ((real_test, REAL_C, "-"), (gen, GEN_C, "--")):
    sub = arr.ravel()[rng.choice(arr.size, min(20000, arr.size), replace=False)]
    axes[2].plot(grid, gaussian_kde(sub)(grid), color=c, ls=ls, lw=1.2)
axes[2].set_yticks([]); axes[2].set_title("value density")

def logspec(x):
    P = np.abs(np.fft.rfft(x, axis=1)) ** 2
    return np.log(P.mean(axis=(0, 2)) + 1e-12)
axes[3].plot(logspec(real_test), color=REAL_C, lw=1.2)
axes[3].plot(logspec(gen), color=GEN_C, ls="--", lw=1.2)
axes[3].set_xlabel("frequency bin"); axes[3].set_title("mean log power spectrum")

for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()

The value density is clipped to the real data's support. A handful of generated
values can land far outside it, and letting those set the axis compresses both
curves into a spike; the fraction left outside is printed below so the clipping
is not hiding anything.

In [ ]:
outside = float(((gen < lo) | (gen > hi)).mean())
print(f"generated values outside the plotted window: {100 * outside:.3f}%")
print(f"real values outside: {100 * float(((real_test < lo) | (real_test > hi)).mean()):.3f}%")

## Scoring another method

`scripts/evaluate_saved.py` runs the same metrics over a folder of
`<dataset>_<suffix>_data.npz` files, so a competing generator is scored by
identical code on identical splits:

```bash
python scripts/evaluate_saved.py --datasets sines --suffix sfmtime --dir results/run/sines_sfmtime
```